In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('/opt/workspace')\n",
    "\n",
    "from datetime import datetime, timedelta\n",
    "from pyspark.sql import SparkSession\n",
    "from pyspark.sql.functions import col, lit, concat_ws, sha2, to_json, struct, current_timestamp\n",
    "from connectors.clickhouse_client import ClickHouseClient\n",
    "from config.settings import oracle_config, clickhouse_config, spark_config, app_config"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ref_date = datetime.now().strftime(\"%Y-%m-%d\")\n",
    "previous_date = (datetime.now() - timedelta(days=1)).strftime(\"%Y-%m-%d\")\n",
    "\n",
    "ref_date, previous_date"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "tables_config = [\n",
    "    {\n",
    "        \"schema\": \"ADMBI_PRD\",\n",
    "        \"table\": \"TABLE_1\",\n",
    "        \"primary_keys\": [\"ID\"]\n",
    "    },\n",
    "    {\n",
    "        \"schema\": \"ADMBI_PRD\",\n",
    "        \"table\": \"TABLE_2\",\n",
    "        \"primary_keys\": [\"CODE\", \"VERSION\"]\n",
    "    }\n",
    "]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "spark = (\n",
    "    SparkSession.builder\n",
    "    .appName(f\"FullPipeline_{ref_date}\")\n",
    "    .config(\"spark.driver.memory\", spark_config.driver_memory)\n",
    "    .config(\"spark.executor.memory\", spark_config.executor_memory)\n",
    "    .config(\"spark.executor.cores\", spark_config.executor_cores)\n",
    "    .config(\"spark.sql.shuffle.partitions\", spark_config.sql_shuffle_partitions)\n",
    "    .config(\"spark.sql.adaptive.enabled\", spark_config.sql_adaptive_enabled)\n",
    "    .config(\"spark.sql.adaptive.coalescePartitions.enabled\", \"true\")\n",
    "    .config(\"spark.jars\", \"/opt/spark/jars/ojdbc11.jar,/opt/spark/jars/clickhouse-jdbc.jar\")\n",
    "    .getOrCreate()\n",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "oracle_jdbc_url = f\"jdbc:oracle:thin:@{oracle_config.host}:{oracle_config.port}/{oracle_config.service}\"\n",
    "protocol = \"https\" if clickhouse_config.secure else \"http\"\n",
    "clickhouse_jdbc_url = f\"jdbc:clickhouse://{protocol}://{clickhouse_config.host}:{clickhouse_config.port}/{clickhouse_config.database}\""
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def ingest_bronze(config):\n",
    "    schema_name = config[\"schema\"]\n",
    "    table_name = config[\"table\"]\n",
    "    primary_keys = config[\"primary_keys\"]\n",
    "    \n",
    "    df_oracle = (\n",
    "        spark.read\n",
    "        .format(\"jdbc\")\n",
    "        .option(\"url\", oracle_jdbc_url)\n",
    "        .option(\"dbtable\", f\"{schema_name}.{table_name}\")\n",
    "        .option(\"user\", oracle_config.user)\n",
    "        .option(\"password\", oracle_config.password)\n",
    "        .option(\"driver\", \"oracle.jdbc.driver.OracleDriver\")\n",
    "        .option(\"fetchsize\", app_config.batch_size)\n",
    "        .option(\"numPartitions\", \"10\")\n",
    "        .load()\n",
    "    )\n",
    "    \n",
    "    primary_key_str = concat_ws(\"|||\", *[col(pk) for pk in primary_keys])\n",
    "    all_columns = [col(c) for c in df_oracle.columns]\n",
    "    row_data = to_json(struct(*all_columns))\n",
    "    row_hash_input = concat_ws(\"|||\", *all_columns)\n",
    "    \n",
    "    df_bronze = df_oracle.select(\n",
    "        lit(ref_date).cast(\"date\").alias(\"ref_date\"),\n",
    "        lit(table_name).alias(\"table_name\"),\n",
    "        primary_key_str.alias(\"primary_key\"),\n",
    "        sha2(row_hash_input, 256).alias(\"row_hash\"),\n",
    "        row_data.alias(\"data\"),\n",
    "        current_timestamp().alias(\"ingestion_timestamp\")\n",
    "    )\n",
    "    \n",
    "    df_bronze.write \\\n",
    "        .format(\"jdbc\") \\\n",
    "        .option(\"url\", clickhouse_jdbc_url) \\\n",
    "        .option(\"dbtable\", \"bronze.snapshot_raw\") \\\n",
    "        .option(\"user\", clickhouse_config.user) \\\n",
    "        .option(\"password\", clickhouse_config.password) \\\n",
    "        .option(\"driver\", \"com.clickhouse.jdbc.ClickHouseDriver\") \\\n",
    "        .option(\"batchsize\", app_config.batch_size) \\\n",
    "        .mode(\"append\") \\\n",
    "        .save()\n",
    "    \n",
    "    return table_name"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "for config in tables_config:\n",
    "    table_name = ingest_bronze(config)\n",
    "    print(f\"Bronze ingestion completed: {table_name}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "spark.stop()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client = ClickHouseClient()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def process_silver(table_name):\n",
    "    client.execute_query(f\"\"\"\n",
    "        INSERT INTO silver.delta_events (ref_date, table_name, primary_key, operation_type, row_hash_after, data_after)\n",
    "        SELECT toDate('{ref_date}'), '{table_name}', current.primary_key, 'INSERT', current.row_hash, current.data\n",
    "        FROM bronze.snapshot_raw AS current\n",
    "        LEFT JOIN bronze.snapshot_raw AS previous ON current.primary_key = previous.primary_key \n",
    "            AND current.table_name = previous.table_name AND previous.ref_date = toDate('{previous_date}')\n",
    "        WHERE current.ref_date = toDate('{ref_date}') AND current.table_name = '{table_name}' \n",
    "            AND previous.primary_key IS NULL\n",
    "    \"\"\")\n",
    "    \n",
    "    client.execute_query(f\"\"\"\n",
    "        INSERT INTO silver.delta_events (ref_date, table_name, primary_key, operation_type, \n",
    "            row_hash_before, row_hash_after, data_before, data_after)\n",
    "        SELECT toDate('{ref_date}'), '{table_name}', current.primary_key, 'UPDATE',\n",
    "            previous.row_hash, current.row_hash, previous.data, current.data\n",
    "        FROM bronze.snapshot_raw AS current\n",
    "        INNER JOIN bronze.snapshot_raw AS previous ON current.primary_key = previous.primary_key\n",
    "            AND current.table_name = previous.table_name AND previous.ref_date = toDate('{previous_date}')\n",
    "        WHERE current.ref_date = toDate('{ref_date}') AND current.table_name = '{table_name}'\n",
    "            AND current.row_hash != previous.row_hash\n",
    "    \"\"\")\n",
    "    \n",
    "    client.execute_query(f\"\"\"\n",
    "        INSERT INTO silver.delta_events (ref_date, table_name, primary_key, operation_type, row_hash_before, data_before)\n",
    "        SELECT toDate('{ref_date}'), '{table_name}', previous.primary_key, 'DELETE', previous.row_hash, previous.data\n",
    "        FROM bronze.snapshot_raw AS previous\n",
    "        LEFT JOIN bronze.snapshot_raw AS current ON previous.primary_key = current.primary_key\n",
    "            AND previous.table_name = current.table_name AND current.ref_date = toDate('{ref_date}')\n",
    "        WHERE previous.ref_date = toDate('{previous_date}') AND previous.table_name = '{table_name}'\n",
    "            AND current.primary_key IS NULL\n",
    "    \"\"\")\n",
    "    \n",
    "    client.execute_query(f\"\"\"\n",
    "        INSERT INTO silver.current_state (table_name, primary_key, row_hash, data, \n",
    "            first_seen_date, last_seen_date, is_active)\n",
    "        SELECT table_name, primary_key, row_hash, data, ref_date, ref_date, 1\n",
    "        FROM bronze.snapshot_raw\n",
    "        WHERE ref_date = toDate('{ref_date}') AND table_name = '{table_name}'\n",
    "    \"\"\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "for config in tables_config:\n",
    "    process_silver(config[\"table\"])\n",
    "    print(f\"Silver processing completed: {config['table']}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client.execute_query(f\"\"\"\n",
    "    INSERT INTO gold.daily_change_metrics (ref_date, table_name, total_inserts, total_updates, total_deletes, total_active_records)\n",
    "    SELECT ref_date, table_name,\n",
    "        countIf(operation_type = 'INSERT'), countIf(operation_type = 'UPDATE'), countIf(operation_type = 'DELETE'),\n",
    "        (SELECT count() FROM silver.current_state AS cs WHERE cs.table_name = de.table_name AND cs.is_active = 1)\n",
    "    FROM silver.delta_events AS de\n",
    "    WHERE ref_date = toDate('{ref_date}')\n",
    "    GROUP BY ref_date, table_name\n",
    "\"\"\")\n",
    "\n",
    "client.execute_query(f\"\"\"\n",
    "    INSERT INTO gold.data_quality_metrics (ref_date, table_name, null_count, duplicate_count, total_records, quality_score)\n",
    "    SELECT toDate('{ref_date}'), table_name,\n",
    "        countIf(data = '' OR data IS NULL), count() - uniq(primary_key), count(),\n",
    "        100.0 * (1 - (countIf(data = '' OR data IS NULL) + count() - uniq(primary_key)) / count())\n",
    "    FROM bronze.snapshot_raw\n",
    "    WHERE ref_date = toDate('{ref_date}')\n",
    "    GROUP BY table_name\n",
    "\"\"\")\n",
    "\n",
    "print(\"Gold aggregation completed\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "summary = client.execute_query_with_result(f\"\"\"\n",
    "    SELECT * FROM gold.daily_change_metrics WHERE ref_date = toDate('{ref_date}')\n",
    "\"\"\")\n",
    "summary.result_rows"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client.close()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.14"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}